# Employee Turnover Analytics
## Portobello Tech — HR Department ML Pipeline

This notebook implements the complete 7-step machine learning pipeline to predict
employee turnover and recommend targeted retention strategies.

### Pipeline Steps
1. **Data Quality Checks** — missing values, types, duplicates, value ranges
2. **Exploratory Data Analysis** — correlation heatmap, distributions, bar charts
3. **Clustering** — K-Means (k=3) on employees who left
4. **Preprocessing & Class Imbalance** — encoding, stratified split, SMOTE
5. **Model Training** — Logistic Regression, Random Forest, Gradient Boosting (5-fold CV)
6. **Model Evaluation** — ROC/AUC, confusion matrices, best model selection
7. **Retention Strategies** — risk zones and targeted recommendations

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 1 — Environment Setup & Imports
# ─────────────────────────────────────────────────────────────────────────────
import sys
import warnings
from pathlib import Path

# Add project root to path so src/ is importable
PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

warnings.filterwarnings('ignore')

import matplotlib
matplotlib.use('Agg')   # Headless backend — saves figures to disk
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Project modules
from src.utils.logger_config import setup_logging
from src.utils.exceptions import EmployeeTurnoverError
from src.data_quality.data_quality_checker import DataQualityChecker
from src.eda.exploratory_analyzer import ExploratoryAnalyzer
from src.clustering.employee_clusterer import EmployeeClusterer
from src.preprocessing.data_preprocessor import DataPreprocessor
from src.modeling.model_trainer import ModelTrainer
from src.modeling.model_evaluator import ModelEvaluator
from src.retention.retention_advisor import RetentionAdvisor, ZONE_ORDER

# ─── Paths ───────────────────────────────────────────────────────────────────
DATA_PATH   = PROJECT_ROOT / 'Dataset' / 'HR_comma_sep.csv'
OUTPUT_DIR  = PROJECT_ROOT / 'outputs'
LOG_DIR     = OUTPUT_DIR / 'logs'
LOG_DIR.mkdir(parents=True, exist_ok=True)

# ─── Logging ─────────────────────────────────────────────────────────────────
logger = setup_logging(log_dir=LOG_DIR)

print(f'Project root : {PROJECT_ROOT}')
print(f'Data path    : {DATA_PATH}')
print(f'Output dir   : {OUTPUT_DIR}')
print(f'Log dir      : {LOG_DIR}')

---
## Step 1: Data Quality Checks

Before any analysis, we validate the raw dataset:
- **Missing values** — null counts and percentages per column
- **Data types** — verify numeric vs categorical columns
- **Duplicates** — identify exact duplicate rows
- **Value ranges** — satisfaction/evaluation must be [0,1]; left/accident must be {0,1}

In [ ]:
# ─── Load Data ────────────────────────────────────────────────────────────────
checker = DataQualityChecker(str(DATA_PATH))
df = checker.load_data()

print(f'Dataset shape: {df.shape}')
print(f'\nColumn names: {list(df.columns)}')
print(f'\nFirst 5 rows:')
df.head()

In [ ]:
# ─── Missing Values ───────────────────────────────────────────────────────────
missing = checker.check_missing_values()
print('Missing Values Analysis:')
print(missing)
print(f'\nTotal missing values: {missing["missing_count"].sum()}')

In [ ]:
# ─── Data Types ──────────────────────────────────────────────────────────────
dtypes = checker.check_data_types()
print('Column Data Types:')
print(dtypes)

In [ ]:
# ─── Duplicates & Value Ranges ───────────────────────────────────────────────
dup_count = checker.check_duplicates()
print(f'Duplicate rows: {dup_count}')

ranges = checker.check_value_ranges()
print('\nValue Range Validation:')
for col, result in ranges.items():
    status = '✓ VALID' if result['valid'] else '✗ INVALID'
    print(f'  {col:30s}: {status}  (invalid_count={result["invalid_count"]})')

In [ ]:
# ─── Full Quality Report ─────────────────────────────────────────────────────
report = checker.generate_quality_report()
print('Quality Report Summary')
print('=' * 40)
print(f'  Total rows         : {report["total_rows"]}')
print(f'  Total columns      : {report["total_columns"]}')
print(f'  Duplicate rows     : {report["duplicate_count"]}')
print(f'  Cols with missing  : {(report["missing_values"]["missing_count"] > 0).sum()}')
print(f'  All ranges valid   : {all(v["valid"] for v in report["value_ranges"].values())}')

---
## Step 2: Exploratory Data Analysis (EDA)

We generate four required visualisations:
- **2.1** Correlation heatmap of all numeric features
- **2.2** Distribution plots for satisfaction_level, last_evaluation, average_montly_hours
- **2.3** Bar chart: project count segmented by left/stayed

All plots are saved to `outputs/plots/eda/`.

In [ ]:
# ─── EDA Initialisation ───────────────────────────────────────────────────────
EDA_OUTPUT = OUTPUT_DIR / 'plots' / 'eda'
analyzer = ExploratoryAnalyzer(df, output_dir=EDA_OUTPUT)

print(f'EDA outputs will be saved to: {EDA_OUTPUT}')

In [ ]:
# ─── 2.1 Correlation Heatmap ─────────────────────────────────────────────────
analyzer.plot_correlation_heatmap()
print('Correlation heatmap saved.')

# Display inline
from IPython.display import Image, display
display(Image(filename=str(EDA_OUTPUT / 'correlation_heatmap.png')))

In [ ]:
# ─── 2.2 Distribution Plots ───────────────────────────────────────────────────
analyzer.plot_all_distributions()
print('Distribution plots saved.')

display(Image(filename=str(EDA_OUTPUT / 'all_distributions.png')))

In [ ]:
# ─── 2.3 Project Count Bar Chart ─────────────────────────────────────────────
analyzer.plot_project_count_bar()
print('Project count bar chart saved.')

display(Image(filename=str(EDA_OUTPUT / 'project_count_bar.png')))

print('\nInference: Employees with only 2 projects or 6-7 projects show the')
print('highest turnover rates, indicating under-utilisation or burn-out.')

In [ ]:
# ─── Turnover Rate by Feature ─────────────────────────────────────────────────
print('Turnover rate by salary:')
print(analyzer.compute_turnover_rate_by_feature('salary'))

print('\nTurnover rate by department (sales):')
print(analyzer.compute_turnover_rate_by_feature('sales'))

---
## Step 3: Clustering of Employees Who Left

We apply **K-Means clustering (k=3)** exclusively to employees who left
(left==1), using two features: `satisfaction_level` and `last_evaluation`.

Expected cluster archetypes:
- **Burned-Out High Performers** — high evaluation, low satisfaction
- **Disengaged Low Performers** — low evaluation, low satisfaction  
- **Poached / Better Opportunity** — high evaluation, high satisfaction

In [ ]:
# ─── K-Means Clustering ──────────────────────────────────────────────────────
CLUSTER_OUTPUT = OUTPUT_DIR / 'plots' / 'clustering'
clusterer = EmployeeClusterer(df, n_clusters=3, output_dir=CLUSTER_OUTPUT)

results = clusterer.run_clustering_pipeline()
print('Clustering complete.')
print(f'\nCluster Summary:')
print(results['summary'])

In [ ]:
# ─── Cluster Interpretation ───────────────────────────────────────────────────
print('Cluster Interpretation:')
print('=' * 60)
for cluster_id, info in results['interpretation'].items():
    print(f'\nCluster {cluster_id}: {info["label"]}')
    print(f'  Satisfaction mean : {info["satisfaction_mean"]}')
    print(f'  Evaluation mean   : {info["evaluation_mean"]}')
    print(f'  Count             : {info["count"]}')
    print(f'  Description: {info["description"][:100]}...')

display(Image(filename=str(CLUSTER_OUTPUT / 'kmeans_clusters.png')))

---
## Step 4: Handle Class Imbalance (SMOTE)

The dataset has a class imbalance — approximately 24% of employees left.
We address this with three preprocessing steps:
1. **One-hot encode** categorical columns (sales, salary) using pd.get_dummies
2. **Stratified 80:20 split** (random_state=123) preserving class proportions
3. **SMOTE** oversampling on the training set only

In [ ]:
# ─── Preprocessing & SMOTE ────────────────────────────────────────────────────
preprocessor = DataPreprocessor(df, target_col='left', test_size=0.2, random_state=123)

# Show class distribution before
print('Class distribution BEFORE SMOTE:')
print(f'  stayed (0): {(df["left"]==0).sum()} ({(df["left"]==0).mean()*100:.1f}%)')
print(f'  left   (1): {(df["left"]==1).sum()} ({(df["left"]==1).mean()*100:.1f}%)')

In [ ]:
# ─── Run Full Preprocessing Pipeline ─────────────────────────────────────────
X_train, X_test, y_train, y_test = preprocessor.run_preprocessing_pipeline()

print('\nAfter preprocessing:')
print(f'  X_train shape  : {X_train.shape}')
print(f'  X_test shape   : {X_test.shape}')
print(f'  Feature columns: {len(X_train.columns)}')

print('\nClass distribution AFTER SMOTE (training set):')
print(f'  stayed (0): {(y_train==0).sum()}')
print(f'  left   (1): {(y_train==1).sum()}')

---
## Step 5: Model Training with 5-Fold Cross-Validation

We train three classifiers on the SMOTE-balanced training data,
each evaluated with **5-fold stratified cross-validation** using F1 as the
primary CV metric.

Models:
- **Logistic Regression** — interpretable linear baseline
- **Random Forest** — ensemble of decision trees, provides feature importance
- **Gradient Boosting** — sequential boosting, typically highest accuracy

In [ ]:
# ─── Train All Models & Save to outputs/models/ ───────────────────────────────
import joblib

MODEL_OUTPUT = OUTPUT_DIR / 'plots' / 'modeling'
MODELS_DIR   = OUTPUT_DIR / 'models'
MODELS_DIR.mkdir(parents=True, exist_ok=True)

trainer = ModelTrainer(X_train, y_train, cv_folds=5, output_dir=MODEL_OUTPUT)

print('Training models (this may take 1-2 minutes) ...')
models = trainer.train_all_models()
print(f'\nTrained models: {list(models.keys())}')

# Save every trained model to outputs/models/
print('\nSaving models to disk ...')
for name, model in models.items():
    safe_name = name.lower().replace(' ', '_')
    path = MODELS_DIR / f'{safe_name}.joblib'
    joblib.dump(model, path)
    print(f'  Saved: {path.name}')


In [ ]:
# ─── CV Scores Summary ────────────────────────────────────────────────────────
print('5-Fold CV F1 Scores:')
print('-' * 50)
for name, model in models.items():
    scores = trainer.get_cv_scores(model)
    print(f'{name:25s}: {scores.mean():.4f} ± {scores.std():.4f}')

In [ ]:
# ─── Classification Reports ──────────────────────────────────────────────────
for name, model in models.items():
    trainer.plot_classification_report(model, X_test, y_test, name)
    print(f'Classification report saved for: {name}')

# Display Gradient Boosting report as example
display(Image(filename=str(MODEL_OUTPUT / 'classification_report_gradient_boosting.png')))

---
## Step 6: Model Evaluation

We evaluate all three models on the held-out test set:
- **6.1** ROC/AUC curves (all models overlaid)
- **6.2** Confusion matrices per model
- **6.3** Select best model and justify Recall over Precision

In [ ]:
# ─── Model Evaluation Setup ──────────────────────────────────────────────────
evaluator = ModelEvaluator(models, X_test, y_test, output_dir=MODEL_OUTPUT)

In [ ]:
# ─── 6.1 ROC / AUC Curves ────────────────────────────────────────────────────
evaluator.plot_roc_curves()
print('ROC curves saved.')

display(Image(filename=str(MODEL_OUTPUT / 'roc_curves.png')))

In [ ]:
# ─── 6.2 Confusion Matrices ───────────────────────────────────────────────────
for name in models:
    evaluator.plot_confusion_matrix(name)
    print(f'Confusion matrix saved for: {name}')

# Display all confusion matrices
for name in models:
    safe_name = name.lower().replace(' ', '_')
    print(f'\n{name}:')
    display(Image(filename=str(MODEL_OUTPUT / f'confusion_matrix_{safe_name}.png')))

In [ ]:
# ─── Evaluation Report Table ─────────────────────────────────────────────────
eval_report = evaluator.generate_evaluation_report()
print('Model Performance Comparison (sorted by AUC):')
print(eval_report.to_string(index=False))

In [ ]:
# ─── Best Model Selection & Save ─────────────────────────────────────────────
best_name, best_model = evaluator.identify_best_model()
print(f'Best model: {best_name}')

# Save best model separately for easy future loading
best_path = MODELS_DIR / 'best_model.joblib'
joblib.dump(best_model, best_path)
print(f'Best model saved: {best_path.name}')
print(f'\nAll models in {MODELS_DIR}:')
for f in sorted(MODELS_DIR.glob('*.joblib')):
    print(f'  {f.name}')


In [ ]:
# ─── 6.3 Recall vs Precision Justification ───────────────────────────────────
print(evaluator.justify_recall_over_precision())

---
## Step 7: Retention Strategies

Using the best model, we:
1. Predict turnover probability for every employee in the test set
2. Assign each to one of four risk zones
3. Generate targeted HR retention strategies per zone

| Zone | Score | Priority |
|------|-------|----------|
| Safe (Green) | < 20% | Monitor |
| Low-Risk (Yellow) | 20–60% | Proactive |
| Medium-Risk (Orange) | 60–90% | Urgent |
| High-Risk (Red) | > 90% | Critical |

In [ ]:
# ─── Retention Advisor ────────────────────────────────────────────────────────
RETENTION_OUTPUT = OUTPUT_DIR / 'plots' / 'retention'
advisor = RetentionAdvisor(best_model, X_test, y_test, output_dir=RETENTION_OUTPUT)

# Predict probabilities
probs = advisor.predict_turnover_probabilities()
print(f'Probability stats — min: {probs.min():.3f}, mean: {probs.mean():.3f}, max: {probs.max():.3f}')

In [ ]:
# ─── Generate Full Retention Report ──────────────────────────────────────────
retention_report = advisor.generate_retention_report()
print(f'Retention report shape: {retention_report.shape}')
print('\nZone distribution:')
print(advisor.get_zone_counts())
print('\nFirst 10 rows (without strategy text):')
retention_report[['employee_index','turnover_probability','risk_zone','actual_left']].head(10)

In [ ]:
# ─── Zone Distribution Plot ───────────────────────────────────────────────────
advisor.plot_zone_distribution()
print('Zone distribution plot saved.')

display(Image(filename=str(RETENTION_OUTPUT / 'risk_zone_distribution.png')))

In [ ]:
# ─── Retention Strategies per Zone ───────────────────────────────────────────
for zone in ZONE_ORDER:
    count = advisor.get_zone_counts().get(zone, 0)
    print(f'\n{"="*60}')
    print(f'{zone.upper()} ZONE — {count} employees')
    print('=' * 60)
    print(advisor.suggest_strategies(zone))

In [ ]:
# ─── Save Retention Report to CSV ────────────────────────────────────────────
report_path = OUTPUT_DIR / 'retention_report.csv'
retention_report.drop(columns=['strategy']).to_csv(report_path, index=False)
print(f'Retention report saved to: {report_path}')

---
## Summary & Conclusions

### Key Findings

1. **Data Quality**: The dataset is clean with no missing values. Duplicate rows may exist but do not affect model integrity significantly.

2. **EDA Insights**:
   - Employees with very few (2) or very many (6-7) projects have the highest turnover rates
   - Low satisfaction combined with high evaluation is a strong departure signal
   - Low salary employees show higher turnover rates

3. **Clustering**: Three distinct archetypes among departing employees:
   - **Burned-Out High Performers** — need workload relief and recognition
   - **Disengaged Low Performers** — need performance support or managed exits
   - **Poached / Better Opportunity** — need competitive compensation and growth paths

4. **Class Imbalance**: SMOTE successfully balanced the training set without
   distorting the test set evaluation.

5. **Model Performance**: Gradient Boosting typically achieves the highest AUC
   on structured HR data. All models outperform the random baseline significantly.

6. **Metric Choice**: **Recall** is the primary metric because missing an employee
   who will leave (false negative) costs 50-200% of their annual salary in
   replacement costs, vastly outweighing the cost of a false positive
   (unnecessary retention conversation).

7. **Retention Strategy**: Risk-zone segmentation enables HR to prioritise
   interventions — focusing immediate resources on Medium-Risk and High-Risk
   employees while maintaining engagement for Low-Risk and Safe employees.

### All Outputs Saved
- Plots: `outputs/plots/`
- Retention report: `outputs/retention_report.csv`
- Logs: `outputs/logs/`